# yt-clips Colab Bridge — Premium v2
## Waits for jobs from your local machine

**Runtime → Change runtime type → T4 GPU**

After this runs, use `./automate.sh` on your Mac with option 2.

In [ ]:
# ════════════════════════════════════
# CELL 1: System + Python Dependencies
# ════════════════════════════════════
import sys, os, subprocess, json, time, glob, shutil
from pathlib import Path

print("Installing system deps...")
!apt-get update -qq && apt-get install -y -qq aria2 ffmpeg 

print("Installing Deno (bot bypass)...")
!curl -fsSL https://deno.land/x/install/install.sh | sh 
os.environ["PATH"] += ":/root/.deno/bin"

print("Installing Python deps...")
!pip install -q yt-dlp faster-whisper rich PyYAML opencv-python-headless numpy filterpy scipy \
    google-api-python-client google-auth-httplib2 google-auth-oauthlib requests Pillow \
    pytest pytest-timeout 

print("Installing Premium GPU deps...")
!pip install -q ultralytics torch --extra-index-url https://download.pytorch.org/whl/cu121 
!pip install -q gfpgan basicsr 

!nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null && echo "GPU OK" || echo "WARNING: No GPU!"
print("\nALL DEPS INSTALLED ✅")

In [ ]:
# ════════════════════════════════════
# CELL 2: Google Drive Auth + Code Sync
# ════════════════════════════════════
from google.colab import auth
auth.authenticate_user()
print("✅ Authenticated")

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io

drive = build('drive', 'v3', cache_discovery=False)

# Find yt-clips folder in Drive
results = drive.files().list(
    q="name='yt-clips' and mimeType='application/vnd.google-apps.folder'",
    spaces='drive', fields='files(id, name)'
).execute()
folders = results.get('files', [])

if not folders:
    print("❌ No 'yt-clips' folder found in Drive.")
    print("   Run './automate.sh' → option 3 on your Mac first to sync code.")
else:
    FOLDER_ID = folders[0]['id']
    print(f"📁 Found yt-clips folder: {FOLDER_ID}")

    # Download all .py files and utils/
    query = f"'{FOLDER_ID}' in parents and (name contains '.py' or name contains '.yaml' or name contains '.json' or name contains '.sh' or name contains '.txt' or name contains '.png')"
    files = drive.files().list(q=query, spaces='drive', fields='files(id, name, mimeType)').execute()
    
    for f in files.get('files', []):
        print(f"  Downloading {f['name']}...")
        request = drive.files().get_media(fileId=f['id'])
        fh = io.BytesIO()
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
        with open(f['name'], 'wb') as out:
            out.write(fh.getvalue())
    
    # Download utils/ folder files
    os.makedirs('utils', exist_ok=True)
    utils_results = drive.files().list(
        q=f"'{FOLDER_ID}' in parents and mimeType='application/vnd.google-apps.folder' and name='utils'",
        spaces='drive'
    ).execute()
    if utils_results.get('files'):
        utils_id = utils_results['files'][0]['id']
        ufiles = drive.files().list(
            q=f"'{utils_id}' in parents", spaces='drive', fields='files(id, name)'
        ).execute()
        for uf in ufiles.get('files', []):
            print(f"  Downloading utils/{uf['name']}...")
            request = drive.files().get_media(fileId=uf['id'])
            fh = io.BytesIO()
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done:
                _, done = downloader.next_chunk()
            with open(f'utils/{uf["name"]}', 'wb') as out:
                out.write(fh.getvalue())
    
    print("✅ Code synced from Drive")

In [ ]:
# ════════════════════════════════════
# CELL 3: Write Config
# ════════════════════════════════════
for folder in ['input', 'temp', 'transcripts', 'highlights', 'shorts', 'logs']:
    os.makedirs(folder, exist_ok=True)

config = """
paths:
  input:       input/
  temp:        temp/
  transcripts: transcripts/
  highlights:  highlights/
  shorts:      shorts/
  logs:        logs/
download:
  format: "bv*+ba/b"
  concurrent_fragments: 32
  use_aria2c: true
  output_filename: "video.mp4"
transcription:
  model: "small"
  language: "hi"
  device: "cuda"
  compute_type: "float16"
highlight:
  audio_energy_threshold: 0.3
  min_duration: 15
  max_duration: 20
  merge_gap: 8
  max_clips: 20
  fast_speech_wpm: 140
  silence_penalty_seconds: 1.5
  use_ai_refinement: true
premium:
  enabled: true
  face_enhancement: true
  frame_interpolation: true
  host_ref_photos: ""
layout:
  has_facecam: true
  facecam: {x: 0, y: 540, width: 320, height: 180}
  facecam_output_height: 400
  gameplay_output_height: 1520
  has_chat_overlay: true
  chat:
    side: "right"
    estimated_width: 350
    brightness_threshold: 30
export:
  width: 1080
  height: 1920
  fps: 60
  video_bitrate: "25M"
  audio_bitrate: "320k"
  crf: 18
  encoder: "h264_nvenc"
  enable_variable_speed: true
  crop_smooth_factor: 0.2
youtube:
  privacy_status: "private"
  category_id: "17"
  upload_enabled: false
  schedule_interval_hours: 2
  niche: "Cricket"
ai:
  provider: "gemini"
  api_key: ""
  model: "gemini-2.0-flash-lite"
  image_model: "gemini-2.5-flash-image"
thumbnail:
  enabled: true
  use_ai: true
  template_path: "channel_logo.png"
  font_size: 120
quality:
  black_threshold: 20
  silence_threshold_db: -35
  frame_sample_count: 5
testing:
  enabled: false
  timeout_seconds: 120
  fail_fast: true
logging:
  level: "INFO"
  log_file: "logs/pipeline.log"
"""
with open('config.yaml', 'w') as f:
    f.write(config.strip())

# Copy AI_API_KEY from Colab secrets if available
from google.colab import userdata
try:
    os.environ['AI_API_KEY'] = userdata.get('AI_API_KEY')
    print("✅ AI_API_KEY loaded from Colab secrets")
except:
    print("⚠️  No AI_API_KEY secret — paste your Gemini key in the cell below")

print("✅ Config written (premium mode)")

In [ ]:
# ════════════════════════════════════
# CELL 4: Run Pre-Flight Tests
# ════════════════════════════════════
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-x", "--timeout=120", "-q"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr[-500:])
    print("\n❌ TESTS FAILED — check code sync")
else:
    print("\n✅ ALL TESTS PASSED")

In [ ]:
# ════════════════════════════════════
# CELL 5: Bridge Worker — Waits for Jobs
# ════════════════════════════════════
print("=" * 50)
print("  🏏 COLAB BRIDGE WORKER ACTIVE")
print("=" * 50)
print()
print("Now go to your Mac and run:")
print("  ./automate.sh \"https://youtu.be/YOUR_VIDEO_ID\"")
print("  → Select option 2 (Remote Run)")
print()
print("Checking for jobs every 30 seconds...")
print()

WATCH_DIR = '/content'
JOB_FILE = 'remote_job.json'
RESULT_FILE = 'remote_job_result.json'

while True:
    job_path = Path(WATCH_DIR) / JOB_FILE
    if job_path.exists():
        try:
            with open(job_path) as f:
                job = json.load(f)
            url = job.get('url', '')
            flags = job.get('flags', [])
            print(f"\n📥 JOB RECEIVED: {url}")
            print(f"   Flags: {flags}")
            
            # Run pipeline
            cmd = [sys.executable, 'pipeline.py', url] + flags
            result = subprocess.run(cmd, capture_output=False, text=True)
            
            # Write result
            with open(Path(WATCH_DIR) / RESULT_FILE, 'w') as f:
                json.dump({
                    'status': 'done' if result.returncode == 0 else 'failed',
                    'returncode': result.returncode,
                    'url': url,
                }, f)
            
            # Remove job file
            job_path.unlink()
            print(f"✅ Job complete (exit={result.returncode})")
        except Exception as e:
            print(f"❌ Job error: {e}")
            job_path.unlink(missing_ok=True)
    
    time.sleep(30)